# Sample Summary

Tracks how many galaxies survive each stage of the pipeline, from the raw SAMI Hα sample through to the final master catalogue.

**Purpose:** a single reference table for reporting sample sizes (e.g. in a thesis methods section) without re-deriving them from each stage's output by hand.

**Data:** counts unique CATIDs directly from each stage's own output on disk - the raw Hα cubes (`data/raw/`), the continuum-masked maps (`data/masked/`), Wang's `SAMI_CIndex_data` (`data/catalogues/SAMI_CIndex_data/`, via `src/utils/wang.py`), and the most recent `outputs/csv/master_<date>.csv` (written by `04_build_master_csv.py`). Nothing here is recomputed - it's a read-only tally of existing pipeline outputs.

**Not a strict funnel:** rows 1-3 (raw → masked → Wang-classified) are nested subsets of each other. Rows 4-7 (per-metric counts) are *not* nested within row 3 or each other - Concentration is read directly from Wang's catalogue (so it tracks row 3 closely), while Gini/Asymmetry/Smoothness additionally require a usable masked Hα map (row 2), so a galaxy can be missing from one metric and present in another. Row 8 is their intersection, row 9 their union.

In [8]:
import re
import sys
from datetime import date
from pathlib import Path

import pandas as pd

# Jupyter's default cwd is the notebook's own directory (sami_sf_pipeline/notebooks/).
PIPELINE_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
assert (PIPELINE_ROOT / "outputs" / "csv").exists(), (
    f"Couldn't find outputs/csv under {PIPELINE_ROOT} - "
    "run this notebook from sami_sf_pipeline/notebooks/, or adjust PIPELINE_ROOT above."
)
sys.path.insert(0, str(PIPELINE_ROOT))

from src.utils.wang import load_all_wang_files

RAW_HALPHA_DIR = PIPELINE_ROOT / "data" / "raw" / "sami_dr3_Halpha_maps"
MASKED_HALPHA_DIR = PIPELINE_ROOT / "data" / "masked" / "halpha"
CSV_DIR = PIPELINE_ROOT / "outputs" / "csv"

CATID_RE = re.compile(r"^(\d+)_")


def unique_catids(directory: Path) -> set[int]:
    """CATIDs from filenames like '100162_A_Halpha_default_1-comp.fits' -
    dedupes multi-component variants and any '<name> 2.fits' OS-duplicate
    copies on disk, since both share the same leading CATID."""
    ids = set()
    for f in directory.iterdir():
        m = CATID_RE.match(f.name)
        if m:
            ids.add(int(m.group(1)))
    return ids

## Stage 1-2: raw sample and continuum-masked sample

Counted directly from filenames on disk - no CSV involved yet at this stage.

In [9]:
raw_ids = unique_catids(RAW_HALPHA_DIR)
masked_ids = unique_catids(MASKED_HALPHA_DIR)

print(f"Full SAMI H\u03b1 sample (data/raw)      : {len(raw_ids)}")
print(f"Continuum-masked (data/masked)      : {len(masked_ids)}")
print(f"Masked but not in raw (unexpected)  : {len(masked_ids - raw_ids)}")

Full SAMI Hα sample (data/raw)      : 3068
Continuum-masked (data/masked)      : 2340
Masked but not in raw (unexpected)  : 0


## Stage 3: Wang classification (`SAMI_CIndex_data`)

Union of CATIDs across Wang's four catalogue files (`SF_Cluster`, `SF_GAMA`, `Composite_Cluster`, `Composite_GAMA`), loaded via the same `src/utils/wang.py` loader `03_metrics/concentration.py` and `04_build_master_csv.py` use - so this number matches what the rest of the pipeline treats as "classified by Wang", sentinel-cleaning included.

In [10]:
wang_files = load_all_wang_files()
for label, df in wang_files.items():
    print(f"  {label:<12}: {df['CATID'].nunique()}")

wang_ids = set()
for df in wang_files.values():
    wang_ids |= set(df["CATID"])

print(f"\nUnique CATIDs across all four Wang files : {len(wang_ids)}")

  sf_gama     : 2080
  sf_cluster  : 879
  com_gama    : 1003
  com_cluster : 755

Unique CATIDs across all four Wang files : 2959


## Stages 4-9: per-metric success and the master catalogue

Loaded from the most recent `outputs/csv/master_<date>.csv` (written by `04_build_master_csv.py`), which already carries per-metric `in_C`/`in_G`/`in_A`/`in_S` flags computed against exactly this pipeline run's `concentration.csv`/`gini.csv`/`asymmetry.csv`/`smoothness.csv` - reusing them here keeps this notebook from re-deriving (and risking drifting from) that merge logic.

In [11]:
MASTER_PATTERN = re.compile(r"^master_(\d{4}-\d{2}-\d{2})\.csv$")
candidates = sorted(
    (m.group(1), p) for p in CSV_DIR.glob("master_*.csv")
    if (m := MASTER_PATTERN.match(p.name))
)
if not candidates:
    raise FileNotFoundError(
        f"No master_<date>.csv found in {CSV_DIR} - run 02_continuum_mask.py, "
        "03_metrics/*.py, then 04_build_master_csv.py first."
    )
run_date, master_path = candidates[-1]
master = pd.read_csv(master_path)
print(f"Using {master_path.name} ({len(master)} rows)")

Using master_2026-08-17.csv (3031 rows)


In [12]:
n_C = int(master["in_C"].sum())
n_G = int(master["in_G"].sum())
n_A = int(master["in_A"].sum())
n_S = int(master["in_S"].sum())
n_all_four = int((master["in_C"] & master["in_G"] & master["in_A"] & master["in_S"]).sum())
n_master = int(master["CATID"].nunique())

print(f"Concentration (C) computed : {n_C}")
print(f"Gini (G) computed          : {n_G}")
print(f"Asymmetry (A) computed     : {n_A}")
print(f"Smoothness (S) computed    : {n_S}")
print(f"All four metrics present   : {n_all_four}")
print(f"Final master catalogue     : {n_master}")

Concentration (C) computed : 2959
Gini (G) computed          : 2340
Asymmetry (A) computed     : 2332
Smoothness (S) computed    : 2259
All four metrics present   : 2259
Final master catalogue     : 3031


## Stage 10: Wang (2022) `Full_sam` quality/purity cuts

`concentration.csv` (and hence `in_C`/stage 3 above) is Wang's *candidate* catalogue - it is not already purity-cut. Their published analysis sample (`Full_sam`, 778 galaxies) applies six further sequential cuts on top of it: angular size, inclination, seeing, star-forming activity (EW(H\u03b1)), pure-AGN exclusion, and sSFR. These are computed by `src/utils/wang_quality_cuts.py` and merged into `master_<date>.csv` as one boolean `cut_*` column per criterion plus an overall `in_full_sam` flag - see that module's docstring for definitions, thresholds, and how each cut was validated against Wang's own published removal counts.

**Known gap:** the pure-AGN cut (`cut_not_pure_agn`) is currently stubbed to pass every galaxy - the column that flags pure AGN in `SAMI_CIndex_data` hasn't been located yet (`galclass` is a group-mass bin, not a spectral/AGN classification, despite the name). `in_full_sam` below is therefore over-inclusive by however many pure-AGN galaxies remain until that's fixed.

In [13]:
from src.utils.wang_quality_cuts import CUT_COLUMNS

cut_counts = {col: int(master[col].sum()) for col in CUT_COLUMNS}
n_full_sam = int(master["in_full_sam"].sum())

for col, n in cut_counts.items():
    print(f"{col:<18}: {n} pass")
print(f"{'in_full_sam':<18}: {n_full_sam} pass (all six cuts)")

cut_size          : 2889 pass
cut_inclination   : 2639 pass
cut_seeing        : 2218 pass
cut_star_forming  : 1747 pass
cut_not_pure_agn  : 2414 pass
cut_ssfr          : 1494 pass
in_full_sam       : 837 pass (all six cuts)


## Summary table

In [14]:
summary = pd.DataFrame([
    {"Stage": "Full SAMI H\u03b1 sample", "Description": "Raw H\u03b1 datacube exists (data/raw/sami_dr3_Halpha_maps)", "N": len(raw_ids)},
    {"Stage": "Continuum-masked", "Description": "Passed 02_continuum_mask.py (S/N >= 3 mask built)", "N": len(masked_ids)},
    {"Stage": "Classified by Wang (Cindex_data)", "Description": "In one of Wang et al.'s four SAMI_CIndex_data files", "N": len(wang_ids)},
    {"Stage": "Concentration (C) computed", "Description": "in_C in master_<date>.csv", "N": n_C},
    {"Stage": "Gini (G) computed", "Description": "in_G in master_<date>.csv", "N": n_G},
    {"Stage": "Asymmetry (A) computed", "Description": "in_A in master_<date>.csv", "N": n_A},
    {"Stage": "Smoothness (S) computed", "Description": "in_S in master_<date>.csv", "N": n_S},
    {"Stage": "All four metrics (C+G+A+S)", "Description": "in_C & in_G & in_A & in_S", "N": n_all_four},
    {"Stage": "Final master catalogue", "Description": f"Union of C/G/A/S CATIDs ({master_path.name})", "N": n_master},
    {"Stage": "  \u2192 size (re <= 15\")", "Description": "cut_size", "N": cut_counts["cut_size"]},
    {"Stage": "  \u2192 inclination (ellip <= 0.7)", "Description": "cut_inclination", "N": cut_counts["cut_inclination"]},
    {"Stage": "  \u2192 seeing", "Description": "cut_seeing", "N": cut_counts["cut_seeing"]},
    {"Stage": "  \u2192 star-forming (EW H\u03b1 >= 1\u00c5)", "Description": "cut_star_forming", "N": cut_counts["cut_star_forming"]},
    {"Stage": "  \u2192 not pure AGN", "Description": "cut_not_pure_agn (STUBBED - passes all, see wang_quality_cuts.py)", "N": cut_counts["cut_not_pure_agn"]},
    {"Stage": "  \u2192 sSFR (log(sSFR) > -11.25)", "Description": "cut_ssfr", "N": cut_counts["cut_ssfr"]},
    {"Stage": "Wang (2022) Full_sam (all 6 cuts)", "Description": "in_full_sam - AGN cut pending, see note above", "N": n_full_sam},
])
summary

,Stage,Description,N
0,Full SAMI Hα sample,Raw Hα datacube exists (data/raw/sami_dr3_Halp...,3068
1,Continuum-masked,Passed 02_continuum_mask.py (S/N >= 3 mask built),2340
2,Classified by Wang (Cindex_data),In one of Wang et al.'s four SAMI_CIndex_data ...,2959
3,Concentration (C) computed,in_C in master_<date>.csv,2959
4,Gini (G) computed,in_G in master_<date>.csv,2340
5,Asymmetry (A) computed,in_A in master_<date>.csv,2332
6,Smoothness (S) computed,in_S in master_<date>.csv,2259
7,All four metrics (C+G+A+S),in_C & in_G & in_A & in_S,2259
8,Final master catalogue,Union of C/G/A/S CATIDs (master_2026-08-17.csv),3031
9,"→ size (re <= 15"")",cut_size,2889


## Save

Writes a dated CSV, matching the rest of the pipeline's convention of never overwriting a previous run's output.

In [15]:
out_path = CSV_DIR / f"sample_summary_{date.today().isoformat()}.csv"
summary.to_csv(out_path, index=False)
print(f"Written: {out_path}")

Written: /Users/sophiesinclair/Desktop/sami copy/sami_sf_pipeline/outputs/csv/sample_summary_2026-08-17.csv
